In [51]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

In [52]:
# 1. Load the data from the CSV file into a pandas DataFrame
csv_file_path = '/Users/tvo/Documents/portfolio/college-notes/CSC 180/canada_per_capita_income.csv'

df = pd.read_csv(csv_file_path)
print("Original Data:")
print(df.head())
print("-" * 30)

Original Data:
   year       income
0  1970  3399.299037
1  1971  3768.297935
2  1972  4251.175484
3  1973  4804.463248
4  1974  5576.514583
------------------------------


In [53]:
# Check for Missing Data
missing_cols = df.isna().any()
if missing_cols.any():
    cols_with_missing = missing_cols[missing_cols].index
    for col in cols_with_missing:
        print(f"Data Missing in {col}")
else:
    print("No Missing Data in Any Column")

No Missing Data in Any Column


In [54]:
target = 'income'
features = [col for col in df.columns if col != target]

X = df[features] 
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

In [55]:
# RandomForestRegressor
experience_order = ['Entry', 'Mid', 'Senior']

numeric_features = ['year']
categorical_features = []
ordinal_features = [] 

transformer_steps = [
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features),
    ('ord', OrdinalEncoder(categories=[experience_order]), ordinal_features)
]

preprocessor = ColumnTransformer(transformers=transformer_steps)

pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor()) 
])

param_dist = {
    'model__n_estimators': [50, 100, 150, 200, 300],
    'model__max_depth': [5, 10, 15, 20, None],
    'model__min_samples_leaf': [1, 2, 4, 6],
    'model__max_features': ['sqrt', 1.0]
}

random_search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist, 
    n_iter=20,
    cv=3,
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)

best_params_from_random = random_search.best_params_

param_grid_focused = {
    'model__n_estimators': [best_params_from_random['model__n_estimators']], 
    'model__max_depth': [8, 10, 12], 
    'model__min_samples_leaf': [1, 2, 3], 
    'model__max_features': [best_params_from_random['model__max_features']] 
}

grid_search = GridSearchCV(estimator=pipe, param_grid=param_grid_focused, cv=3, n_jobs=-1)

grid_search.fit(X_train, y_train)

,estimator,Pipeline(step...Regressor())])
,param_grid,"{'model__max_depth': [8, 10, ...], 'model__max_features': [1.0], 'model__min_samples_leaf': [1, 2, ...], 'model__n_estimators': [150]}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...), ...]"


In [56]:
print("--- Model Results ---")
best_pipeline = grid_search.best_estimator_
model = best_pipeline.named_steps['model']
preprocessor = best_pipeline.named_steps['preprocessor']
importances = model.feature_importances_
feature_names = preprocessor.get_feature_names_out()

print("Feature Importances:")
for name, importance in zip(feature_names, importances):
    print(f"{name}: {importance:.4f}")

print("\n--- GridSearch Stats ---")
print(f"Best parameters found: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.4f}") # Uses K-fold to predict
print(f"Test set score: {grid_search.score(X_test, y_test):.4f}")

--- Model Results ---
Feature Importances:
num__year: 1.0000

--- GridSearch Stats ---
Best parameters found: {'model__max_depth': 8, 'model__max_features': 1.0, 'model__min_samples_leaf': 1, 'model__n_estimators': 150}
Best cross-validation score: 0.9721
Test set score: 0.9805


In [57]:
# LinearRegrsssion w/out Dummy Variable
experience_order = ['Entry', 'Mid', 'Senior']

numeric_features = ['year']
categorical_features = []
ordinal_features = [] 

transformer_steps = [
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features),
    ('ord', OrdinalEncoder(categories=[experience_order]), ordinal_features)
]

preprocessor = ColumnTransformer(transformers=transformer_steps)

pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression()) 
])

pipe.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [58]:
print(f"Test set score: {pipe.score(X_test, y_test):.4f}")

model = pipe.named_steps['model']
feature_names = pipe.named_steps['preprocessor'].get_feature_names_out()

print("\nCoefficients:")
for name, coef in zip(feature_names, model.coef_):
    print(f"{name}: {coef:.2f}")

Test set score: 0.8912

Coefficients:
num__year: 11518.70


In [59]:
data_to_predict = pd.DataFrame([
    {'year': 2030},
    {'year': 1965},
])

predictions_rf = grid_search.predict(data_to_predict)
predictions_lg = pipe.predict(data_to_predict)

for year, pred_rf, pred_lg in zip(data_to_predict['year'], predictions_rf, predictions_lg):
    print(f"\nFor the year {year}:")
    print(f"  - Random Forest Predicted Income: ${pred_rf:,.2f}")
    print(f"  - Linear Regression Predicted Income: ${pred_lg:,.2f}")


For the year 2030:
  - Random Forest Predicted Income: $37,565.72
  - Linear Regression Predicted Income: $49,921.96

For the year 1965:
  - Random Forest Predicted Income: $3,604.45
  - Linear Regression Predicted Income: $-3,619.22
